# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**.
- Do **not** directly merge `orders` with `order_items`, `order_payments`, or `order_reviews` and assume the result is still one row per order.
- Aggregate item/payment/review tables to `order_id` first, then merge those summaries into `orders`.
- Check row counts and duplicate `order_id` values after every major merge.
- If your target is `delivered_late`, parse date columns first and think carefully about which fields would be available at prediction time.
- If your target is `review_score` or `high_review_score`, avoid using review text/comment fields as predictors because that would leak information from the outcome process.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
pd.set_option('display.max_columns', 100)

## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP



geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

In [5]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
customer_id,0.00000
order_status,0.00000
order_purchase_timestamp,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


...

,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str
geolocation_records,int64


,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating an order-level ABT, aggregate lower-level tables. Examples include:

- item count, product count, seller count, total item price, total freight by `order_id`
- total payment value, max installments, number of payment types by `order_id`
- average review score or review availability by `order_id`

After merging, validate that `order_id` is not duplicated in the ABT.

In [6]:
item_summary = (
    items.groupby('order_id')
    .agg(
        order_item_count=('order_item_id', 'count'),
        product_count=('product_id', 'nunique'),
        seller_count=('seller_id', 'nunique'),
        item_price_total=('price', 'sum'),
        freight_total=('freight_value', 'sum')
    )
    .reset_index()
)

payment_summary = (
    payments.groupby('order_id')
    .agg(
        payment_total=('payment_value', 'sum'),
        payment_installments_max=('payment_installments', 'max'),
        payment_type_count=('payment_type', 'nunique')
    )
    .reset_index()
)

review_summary = (
    reviews.groupby('order_id')
    .agg(review_score=('review_score', 'mean'))
    .reset_index()
)

abt = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(item_summary, on='order_id', how='left')
    .merge(payment_summary, on='order_id', how='left')
    .merge(review_summary, on='order_id', how='left')
)

abt['delivery_days'] = (
    abt['order_delivered_customer_date'] - abt['order_purchase_timestamp']
).dt.days
abt['estimated_delivery_days'] = (
    abt['order_estimated_delivery_date'] - abt['order_purchase_timestamp']
).dt.days
abt['delivered_late'] = (
    abt['order_delivered_customer_date'] > abt['order_estimated_delivery_date']
).astype('Int64')
abt['high_review_score'] = (abt['review_score'] >= 4).astype('Int64')

In [7]:
print(f'Preliminary ABT shape: {abt.shape}')
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame('missing_rate'))

Preliminary ABT shape: (25000, 25)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_count,product_count,seller_count,item_price_total,freight_total,payment_total,payment_installments_max,payment_type_count,review_score,delivery_days,estimated_delivery_days,delivered_late,high_review_score
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,32ea3bdedab835c3aa6cb68ce66565ef,4106,sao paulo,SP,1.0,1.0,1.0,29.90,15.56,45.46,1,1,4.0,23.0,23,0,1
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,87776adb449c551e74c13fc34f036105,12030,taubate,SP,1.0,1.0,1.0,29.99,10.96,40.95,4,1,5.0,10.0,50,0,1
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,NaN,NaN,NaN,NaN,NaN,50.74,3,1,2.0,NaN,50,0,0
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,1.0,1.0,1.0,249.90,17.59,267.49,10,1,3.0,12.0,54,0,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,1.0,1.0,1.0,67.90,18.98,86.88,1,1,5.0,51.0,54,0,1


,missing_rate
order_delivered_customer_date,0.03100
delivery_days,0.03100
order_delivered_carrier_date,0.01808
order_item_count,0.00836
item_price_total,0.00836
freight_total,0.00836
product_count,0.00836
seller_count,0.00836
review_score,0.00812
order_approved_at,0.00156


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

Choose a target that matches your business question. For late-delivery prediction, consider excluding non-delivered orders because they do not have normal delivered dates. For review-score prediction, avoid predictors that are created after the review is written.

In [8]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,25000,25000,3b697a20d9e427646d92567910af6d57,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,25000,25000,355077684019f7f60a031656bd7262b8,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,25000,8,delivered,24222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,25000,NaN,NaN,NaN,2017-12-31 22:18:29.561480,2016-10-03 09:44:50,2017-09-13 11:52:50,2018-01-19 19:08:12.500000,2018-05-05 11:17:57,2018-10-01 15:30:09,NaN
order_approved_at,24961,NaN,NaN,NaN,2018-01-01 06:46:13.413765,2016-10-04 10:19:23,2017-09-13 19:55:11,2018-01-20 09:08:59,2018-05-05 11:50:40,2018-09-03 17:40:06,NaN
order_delivered_carrier_date,24548,NaN,NaN,NaN,2018-01-05 06:50:25.146529,2016-10-08 10:34:01,2017-09-18 18:01:51.750000,2018-01-24 20:01:44.500000,2018-05-08 14:21:00,2018-09-04 15:25:00,NaN
order_delivered_customer_date,24225,NaN,NaN,NaN,2018-01-14 19:15:33.384024,2016-10-11 14:46:49,2017-09-26 19:04:55,2018-02-03 03:49:26,2018-05-16 12:26:50,2018-09-27 02:24:33,NaN
order_estimated_delivery_date,25000,NaN,NaN,NaN,2018-01-24 17:19:26.976000,2016-10-27 00:00:00,2017-10-04 00:00:00,2018-02-15 00:00:00,2018-05-28 00:00:00,2018-10-23 00:00:00,NaN
customer_unique_id,25000,24764,8d50f5eadf50201ccdcedfb9e2ac8455,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,25000.0,NaN,NaN,NaN,34966.35068,1004.0,11431.75,24326.0,58037.0,99970.0,29706.603087


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

In [9]:
# Save your final ABT when it is ready.
# Path('outputs').mkdir(exist_ok=True)
# abt.to_csv('outputs/final_abt.csv', index=False)